In [ ]:
%pip install --upgrade pip 
%pip install --upgrade transformers datasets[audio] accelerate

In [ ]:
import os
# path to ffmpeg bin directory(C:\LLM_Project\ffmpeg\bin)
os.environ["PATH"] += os.pathsep + r"C:\LLM_Project\ffmpeg\bin"  

In [ ]:
import torch

from transformers import AutoModelForSpeechSeq2Seq
from transformers import AutoProcessor
from transformers import pipeline
#from datasets import load_dataset  # Using Audio_Boohwal.mp3

device = "cuda:0" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

model_id = "openai/whisper-large-v3-turbo"  # you can change to other Whisper models

model = AutoModelForSpeechSeq2Seq.from_pretrained(
    model_id, torch_dtype=torch_dtype, low_cpu_mem_usage=True, use_safetensors=True
)
model.to(device)

processor = AutoProcessor.from_pretrained(model_id)

pipe = pipeline(        #Create a pipeline for speech recognition
    "automatic-speech-recognition",
    model=model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    torch_dtype=torch_dtype,
    device=device,
    return_timestamps=True, # to get word-level timestamps
    chunk_length_s=10,      # long audio files -> chunk size(10 seconds) as needed
    stride_length_s=2,      # overlapping chunks(2 seconds) to avoid missing words   
)

#dataset = load_dataset("distil-whisper/librispeech_long", "clean", split="validation") 
#sample = dataset[0]["audio"]        # example from dataset
#sample = "./Data/Audio_Boohwal.mp3"  # local audio file path
sample = "C:/LLM_Project/Data/STT_Audio_58s.mp3"  # local audio file path

result = pipe(sample)
#print(result["text"])

print(result)   # print full result with word-level timestamps

In [ ]:
# Save Chunks to CSV file
start_end_text = [] # extract start, end, text from each chunk

for chunk in result["chunks"]:
    start = chunk["timestamp"][0]
    end = chunk["timestamp"][1]
    text = chunk["text"]
    start_end_text.append([start, end, text])

import pandas as pd
df = pd.DataFrame(start_end_text, columns=["start", "end", "text"])
df.to_csv("STT_Audio_Chunks.csv", index=False, sep="|")
display(df)